In [ ]:
# SECCION A - Integrante 1: carga, limpieza y construccion de variables (Puntos 1-2)
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyreadr
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, learning_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, roc_auc_score, roc_curve)

random.seed(123)
np.random.seed(123)
##

In [ ]:
# Cargar el dataset directamente desde el archivo RData con pyreadr
contenido_rdata = pyreadr.read_r("listings.RData")
listings = contenido_rdata["listings"]

print("Dimensiones originales:", listings.shape)
listings.head()
##

Dimensiones originales: (171748, 80)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,city
0,5456.0,https://www.airbnb.com/rooms/5456,2.025092e+13,2025-09-17,city scrape,"Walk to 6th, Rainey St and Convention Ctr",Great central location for walking to Convent...,My neighborhood is ideally located if you want...,https://a0.muscache.com/pictures/14084884/b5a3...,8028,...,4.73,4.79,NaN,f,1,1,0,0,3.52,"Austin, Texas"
1,6448.0,https://www.airbnb.com/rooms/6448,2.025092e+13,2025-09-17,city scrape,"Secluded Studio @ Zilker - King Bed, Bright & ...","Clean, private space with everything you need ...",The neighborhood is fun and funky (but quiet)!...,https://a0.muscache.com/pictures/airflow/Hosti...,14156,...,4.97,4.88,NaN,t,1,1,0,0,1.98,"Austin, Texas"
2,8502.0,https://www.airbnb.com/rooms/8502,2.025092e+13,2025-09-17,city scrape,Woodland Studio Lodging,Studio rental on lower level of home located i...,,https://a0.muscache.com/pictures/miso/Hosting-...,25298,...,4.69,4.63,NaN,f,1,1,0,0,0.28,"Austin, Texas"
3,13035.0,https://www.airbnb.com/rooms/13035,2.025092e+13,2025-09-17,city scrape,Historic house in highly walkable East Austin,Comfortable 2 bedroom/2 bathroom home very cen...,East Cesar Chavez is a gentrifying urban area ...,https://a0.muscache.com/pictures/miso/Hosting-...,50793,...,5.00,4.95,NaN,f,2,2,0,0,0.11,"Austin, Texas"
4,22828.0,https://www.airbnb.com/rooms/22828,2.025092e+13,2025-09-16,city scrape,Garage Apartment central SE Austin,"Fully furnished, centrally located, second sto...","wikipedia: East_Riverside-Oltorf,_Austin,_Texas",https://a0.muscache.com/pictures/miso/Hosting-...,56488,...,4.72,4.84,NaN,f,1,1,0,0,0.30,"Austin, Texas"


In [ ]:
# Limpiar precio (quitar $ y comas), forzar columnas numericas, eliminar NA y outliers (p99)
listings["precio_num"] = (
    listings["price"].astype(str).str.replace(r"[\$,]", "", regex=True)
)
listings["precio_num"] = pd.to_numeric(listings["precio_num"], errors="coerce")

columnas_modelo = ["precio_num", "accommodates", "bathrooms", "bedrooms", "beds",
                   "minimum_nights", "number_of_reviews", "review_scores_rating",
                   "availability_365", "reviews_per_month"]

datos = listings[columnas_modelo].copy()
# pyreadr devuelve algunas columnas como object, hay que forzarlas a numerico
for col in columnas_modelo:
    datos[col] = pd.to_numeric(datos[col], errors="coerce")

datos = datos.dropna()
datos = datos[datos["precio_num"] > 0]
limite_precio = datos["precio_num"].quantile(0.99)
datos = datos[datos["precio_num"] <= limite_precio].reset_index(drop=True)

print("Dimensiones despues de limpieza:", datos.shape)
datos["precio_num"].describe() ## 

Dimensiones despues de limpieza: (62095, 10)


count    62095.000000
mean       258.341767
std        257.958774
min          8.000000
25%        116.000000
50%        181.000000
75%        297.000000
max       2112.000000
Name: precio_num, dtype: float64

In [ ]:
# PUNTO 1: variable categorica de precio (economica / media / cara) usando terciles
datos["categoria_precio"] = pd.qcut(
    datos["precio_num"], q=3, labels=["economica", "media", "cara"]
)

# Tres variables dicotomicas solicitadas (valores 0 y 1)
datos["es_cara"]      = (datos["categoria_precio"] == "cara").astype(int)
datos["es_media"]     = (datos["categoria_precio"] == "media").astype(int)
datos["es_economica"] = (datos["categoria_precio"] == "economica").astype(int)

print("Frecuencia por categoria:")
print(datos["categoria_precio"].value_counts())
print("\nProporciones:")
print(datos["categoria_precio"].value_counts(normalize=True).round(3))
datos[["precio_num", "categoria_precio", "es_cara", "es_media", "es_economica"]].head() ##

Frecuencia por categoria:
categoria_precio
economica    20937
cara         20644
media        20514
Name: count, dtype: int64

Proporciones:
categoria_precio
economica    0.337
cara         0.332
media        0.330
Name: proportion, dtype: float64


,precio_num,categoria_precio,es_cara,es_media,es_economica
0,97.0,economica,0,0,1
1,160.0,media,0,1,0
2,38.0,economica,0,0,1
3,145.0,media,0,1,0
4,58.0,economica,0,0,1
